# 第 6 周练习 —— 微调（Fine Tuning）商品定价器

## 练习目标

从 Amazon 商品元数据策展一套「价格均衡」的训练集，先跑多条 **非 LLM 基线**（随机、均值、线性、词袋、Word2Vec、随机森林），再把对话数据写成 JSONL，对 `gpt-4o-mini` 做 **Chat Fine-Tuning**，并用 `Tester` 对比误差。

## 和本课 Week 6 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 数据清洗与 prompt 组装 | `Item.parse` / `make_prompt` |
| 价格桶均衡抽样 | `price_slots` + `balanced_bundle` |
| 基线模型阶梯 | 随机 → 均值 → 特征回归 → NLP → 森林 |
| OpenAI Fine-Tuning | JSONL `messages` + `fine_tuning.jobs.create` |
| 统一评测 | `Tester.test(predictor, data)`（RMSLE / Hits） |

## 怎么跑

1. 先装依赖格（`gensim`、`datasets==3.6.0`），并准备 `.env`：`OPENAI_API_KEY`、`HF_TOKEN`
2. 加载 Amazon 目录会较久（多进程）；可用注释掉的品类控制规模
3. 微调格会真实调用 OpenAI；确认 `fine_tune_train` 条数后再跑


In [ ]:
# ========== 安装本练习额外依赖 ==========

# gensim：Word2Vec 词向量基线需要
!pip install gensim
# 锁定 datasets 3.6.0：与后面 load_dataset / trust_remote_code 行为对齐
!pip install --upgrade datasets==3.6.0


In [ ]:
# ========== 导入：数据策展 + 基线模型 + OpenAI 微调所需工具箱 ==========

# os：环境变量
import os
# math：RMSLE 里用 log / sqrt
import math
# random：打乱、随机基线、采样种子
import random
# json：解析 details 特征 JSON
import json
# pickle：把 Item 列表持久化到磁盘
import pickle
# re：清洗文本、从模型回复抽价格
import re
# numpy：加权抽样、向量均值
import numpy as np
# pandas：描述统计与特征表
import pandas as pd
# tqdm：多进程加载时的进度条
from tqdm import tqdm
# Path：数据目录路径
from pathlib import Path
# OpenAI：Files / Fine-Tuning / Chat
from openai import OpenAI
# datetime：加载耗时统计
from datetime import datetime
# load_dotenv：读 .env
from dotenv import load_dotenv
# matplotlib：Tester 散点图与直方图
import matplotlib.pyplot as plt
# login：Hugging Face Hub
from huggingface_hub import login
# LinearSVR：Word2Vec 特征上的线性支持向量回归
from sklearn.svm import LinearSVR
# Word2Vec：文档向量基线
from gensim.models import Word2Vec
# display：在笔记本里漂亮展示 DataFrame
from IPython.display import display
# AutoTokenizer：用 Llama tokenizer 统计/截断 token
from transformers import AutoTokenizer
# simple_preprocess：gensim 简易分词
from gensim.utils import simple_preprocess
# Counter / defaultdict：品类计数、价格桶
from collections import Counter, defaultdict
# LinearRegression：线性与词袋基线
from sklearn.linear_model import LinearRegression
# RandomForestRegressor：表格特征森林基线
from sklearn.ensemble import RandomForestRegressor
# ProcessPoolExecutor：按 chunk 并行解析商品
from concurrent.futures import ProcessPoolExecutor
# Dataset / DatasetDict / load_dataset：HF datasets
from datasets import Dataset, DatasetDict, load_dataset
# mean_squared_error / r2_score：指标（本笔记本主要靠 Tester）
from sklearn.metrics import mean_squared_error, r2_score
# CountVectorizer：词袋（Bag-of-Words）特征
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
# ========== 环境变量 + Hugging Face 登录 ==========

# 加载 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenAI Key（后面 OpenAI() 默认也会读；这里先取到变量）
openai_key = os.environ.get("OPENAI_API_KEY")

# 可选：Anthropic Key（本练习未启用，仅注释占位）
# 【注】anthropic_key = os.environ.get("ANTHROPIC_API_KEY")

# Hugging Face token：拉 Amazon 数据集 / 推模型可能用到
hf_token = os.environ.get("HF_TOKEN")
# 打印 token（注意：共享笔记本时勿提交含真实 token 的输出）
print(hf_token)

# 有 token 才登录 Hub
if hf_token:
    print("Loggin in...")
    login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 终端彩色输出：Tester 按误差涂色 ==========

# ANSI 转义：绿 / 黄 / 红 / 复位
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
# 把逻辑颜色名映射到 ANSI 码，供 print 使用
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


In [ ]:
# ========== Item：把原始商品 dict 清洗成「问答式定价 prompt」 ==========

# 用 Llama 3.1 8B 的 tokenizer 做长度控制（不要求本机加载完整权重推理）
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

# 正文最短字符数：太短的商品直接丢弃
MIN_CHARS = 300
# 编码后至少这么多 token 才纳入
MIN_TOKENS = 150
# prompt 正文最多保留的 token 数
MAX_TOKENS = 160
# 字符截断上限：约按 7 字符/token 粗估，避免先 encode 超长文本
CEILING_CHARS = MAX_TOKENS * 7

class Item:
    # 类级共享 tokenizer：所有 Item 复用同一份词表
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    # 训练时价格前缀；test_prompt 截到此处让模型续写
    PREFIX = "Price is $"
    # 提问句（影响行为的英文字符串，保持原样）
    QUESTION = "How much does this cost to the nearest dollar?"
    # details 里要删掉的噪声片段（电池、包装、畅销榜等）
    REMOVALS = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', '"Batteries Required?": "No"', '"Batteries Required?": "Yes"', "By Manufacturer", "Item", "Date First", "Package", ":", "Number of", "Best Sellers", "Number", "Product "]

    def __init__(self, data, price):
        # 商品标题
        self.title = data["title"]
        # 标签价格（美元）
        self.price = price
        # 品类：后面均衡抽样会用到
        self.category = data.get("category", "Unknown")
        # 最终 prompt 的 token 数
        self.token_count = 0
        # 原始 details（可能是 JSON 字符串）
        self.details = None
        # 训练用完整 prompt（含价格答案）
        self.prompt = None
        # 是否通过长度/质量门槛
        self.include = False
        # 立刻解析；失败则 include 保持 False
        self.parse(data)

    def scrub_details(self):
        # 从 details 文本中抠掉 REMOVALS 噪声子串
        details = self.details

        for remove in self.REMOVALS:
            details = details.replace(remove, "")

        return details

    def scrub(self, text):
        # 标点/空白归一
        text = re.sub(r'[:\[\]"{}【】\s]+', ' ', text).strip()
        # 清理多余逗号
        text = text.replace(" ,", ",").replace(",,,",",").replace(",,",",")
        words = text.split(" ")
        # 丢掉「很长且含数字」的词（常是型号噪声）；短词一律保留
        select = [word for word in words if len(word) < 7 or not any(char.isdigit() for char in word)]
        return " ".join(select)

    def parse(self, data):
        # description 可能是字符串列表 → 拼成一段
        contents = '\n'.join(data.get("description", []))

        if contents:
            contents += '\n'

        # features 列表同样拼进正文
        features = '\n'.join(data.get("features", []))
        if features:
            contents += features + '\n'

        # details：先存起来，再 scrub 后追加
        self.details = data.get("details")
        if self.details:
            contents += self.scrub_details() + '\n'

        # 太短：直接放弃（include 仍为 False）
        if len(contents) > MIN_CHARS:
            # 先按字符粗截，再精确按 token 截
            contents = contents[:CEILING_CHARS]
            # 标题 + 正文都 scrub 后再拼
            text = f"{self.scrub(self.title)}\n{self.scrub(contents)}"
            tokens = self.tokenizer.encode(text, add_special_tokens=False)

            # token 太少也丢弃；够长则截到 MAX_TOKENS 并 decode 回文本
            if len(tokens) > MIN_TOKENS:
                tokens = tokens[:MAX_TOKENS]
                text = self.tokenizer.decode(tokens)
                self.make_prompt(text)
                self.include = True

    def make_prompt(self, text):
        # 拼训练样本：问题 + 商品文本
        self.prompt = f"{self.QUESTION}\n\n{text}\n\n"
        # 追加真值价格（圆整到美元 + .00）
        self.prompt += f"{self.PREFIX}{str(round(self.price))}.00"
        # 记录整段 prompt 的 token 数
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        # 推理用：去掉价格数字，只留到 PREFIX
        return self.prompt.split(self.PREFIX)[0] + self.PREFIX

    def __repr__(self):
        # 调试打印：标题 = $价格
        return f"<{self.title} = ${self.price}>"


In [ ]:
# ========== ItemLoader：从 HF Amazon-Reviews 元数据并行加载 Item ==========

# 合法价格下界（美元）
MIN_PRICE = 0.5
# 每个并行任务处理的样本块大小
CHUNK_SIZE = 1000
# 合法价格上界（美元）
MAX_PRICE = 999.49

class ItemLoader:
    def __init__(self, name):
        # 品类名：对应 raw_meta_{name} 配置
        self.name = name
        # 稍后 load() 时填入 HF Dataset
        self.dataset = None

    def from_datapoint(self, datapoint):
        # 单条原始记录 → Item 或 None
        try:
            price_str = datapoint.get("price")
            if price_str:
                # 字符串价格转 float
                price = float(price_str)
                # 价格落在合法区间才继续清洗
                if MIN_PRICE <= price <= MAX_PRICE:
                    item = Item(datapoint, price)
                    # 只有通过长度门槛的才返回
                    if item.include:
                        return item
        except ValueError:
            # 价格字段无法 float 时安静跳过
            return None

    def from_chunk(self, chunk):
        # 处理一个 Dataset 切片
        batch = []
        for datapoint in chunk:
            item = self.from_datapoint(datapoint)

            if item:
                batch.append(item)

        return batch

    def chunk_generator(self):
        # 按 CHUNK_SIZE 产出 dataset.select(...) 切片
        size = len(self.dataset)
        for start in range(0, size, CHUNK_SIZE):
            yield self.dataset.select(range(start, min(start + CHUNK_SIZE, size)))

    def load_in_parallel(self, workers):
        # 多进程解析全部 chunk
        results = []
        # tqdm 需要的总 chunk 数（向上取整的粗算）
        chunk_count = (len(self.dataset) // CHUNK_SIZE) + 1

        with ProcessPoolExecutor(max_workers=workers) as pool:
            # pool.map 保序；tqdm 显示 chunk 进度
            for batch in tqdm(pool.map(self.from_chunk, self.chunk_generator()), total=chunk_count):
                results.extend(batch)

        # 用目录名覆盖/填写品类，便于后续按品类统计
        for result in results:
            result.category = self.name

        return results

    def load(self, workers=8):
        # 拉取 McAuley Amazon-Reviews-2023 的 raw_meta_{品类} 全量 split
        self.dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_meta_{self.name}", split="full", trust_remote_code=True)
        start = datetime.now()
        print(f"Loading {self.dataset}")
        # 并行解析
        results = self.load_in_parallel(workers)
        # 耗时（分钟）
        duration = (datetime.now() - start).total_seconds() / 60
        print(f"Completed {self.name} with {len(results):,} items in {duration:.1f} mins")
        return results


In [ ]:
# ========== Tester：统一评测框架（误差着色 + 散点图 + RMSLE） ==========

class Tester:

    def __init__(self, predictor, data, title=None, size=250):
        # 预测函数：输入 Item，输出美元价格（float）
        self.predictor = predictor
        # 测试集（或切片）
        self.data = data
        # 默认标题：函数名 snake_case → Title Case
        self.title = title or predictor.__name__.replace("_", " ").title()
        # 评测条数上限
        self.size = size
        # 累积预测值
        self.guesses = []
        # 累积真值
        self.truths = []
        # 累积绝对误差
        self.errors = []
        # 累积 Squared Log Error
        self.sles = []
        # 累积颜色标签（green/orange/red）
        self.colors = []

    def color_for(self, error, truth):
        # 绿：绝对误差 <40 或相对误差 <20%
        if error < 40 or error / truth < 0.2:
            return "green"

        # 橙：绝对误差 <80 或相对误差 <40%
        if error < 80 or error / truth < 0.4:
            return "orange"

        # 其余涂红
        return "red"

    def run_datapoint(self, index):
        # 取第 index 条商品
        datapoint = self.data[index]
        # 调用预测器
        guess = self.predictor(datapoint)
        truth = datapoint.price
        # 绝对误差（美元）
        error = abs(guess - truth)
        # log 空间误差（+1 避免 log(0)）
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        # 标题过长截断，避免刷屏
        name = datapoint.title if len(datapoint.title) <= 40 else datapoint.title[:40] + "..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        # 彩色一行日志（ANSI）
        print(f"{COLOR_MAP[color]}{index + 1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {name}{RESET}")

    def chart(self, title):
        # 真值 vs 预测散点图
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        # 对角线：完美预测参考线
        plt.plot([0, max_val], [0, max_val], color="deepskyblue", lw=2, alpha=0.6)
        # 点颜色沿用 green/orange/red 字符串（matplotlib 可识别）
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel("Ground Truth")
        plt.ylabel("Model Estimate")
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        # 平均绝对误差
        average_error = sum(self.errors) / self.size
        # Root Mean Squared Log Error
        rmsle = math.sqrt(sum(self.sles) / self.size)
        # 绿色命中数
        hits = sum(1 for color in self.colors if color == "green")
        # 标题嵌入三项指标
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits / self.size * 100:.1f}%"
        self.chart(title)

    def run(self):
        # 逐条评测
        for index in range(self.size):
            self.run_datapoint(index)

        # 汇总出图
        self.report()

    @classmethod
    def test(cls, function, data):
        # 快捷入口：构造实例并 run
        cls(function, data).run()

def get_price(s):
    # 去掉 $ 与千分位逗号
    s = s.replace("$", "").replace(",", "")
    # 抓第一个数字（可含符号/小数）
    match = re.search(r"[-+]?\d*\.?\d+", s)
    return float(match.group()) if match else 0.0


## 数据

从多个 Amazon 品类加载、清洗、按价格桶均衡，再划分 train / test。


### 加载目录 / 商品表

按 `catalog_labels` 逐个 `ItemLoader.load()`，合并进 `curated_pool`。


In [ ]:
# ========== 选择要加载的 Amazon 品类并并行策展 ==========

catalog_labels = [
    "All_Beauty",
    # 下列品类默认注释掉以控制下载/解析时间；需要更大池时可取消注释
    # 【注】"Automotive",
    # 【注】"Electronics",
    # 【注】"Office_Products",
    # 【注】"Tools_and_Home_Improvement",
    # 【注】"Cell_Phones_and_Accessories",
    # 【注】"Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
    "Software",
    "Handmade_Products"
]
# 汇总所有品类的 Item
curated_pool = []

for label in catalog_labels:
    print("Loading " + label)
    loader = ItemLoader(label)
    # extend：把该品类结果追加到总池
    curated_pool.extend(loader.load())

print(f"Total curated items: {len(curated_pool):,}")


In [ ]:
# ========== 快速摸清价格 / token / 品类分布 ==========

# 价格序列、token 数序列
price_series = [item.price for item in curated_pool]
token_series = [item.token_count for item in curated_pool]
# 各品类计数
category_tally = Counter(item.category for item in curated_pool)
# 用 DataFrame 做 describe（均值、分位数等）
summary_frame = pd.DataFrame({"price": price_series, "tokens": token_series})

display(summary_frame.describe())
# 品类按数量降序展示
display(pd.DataFrame.from_dict(category_tally, orient="index", columns=["count"]).sort_values("count", ascending=False))


In [ ]:
# ========== 按「四舍五入到整数美元」分桶 ==========

# key=圆整价格 → 该价格下的 Item 列表
price_slots = defaultdict(list)
for item in curated_pool:
    key = round(item.price)
    # 只要 $1–$999 的桶（与后面 range(1,1000) 对齐）
    if 1 <= key <= 999:
        price_slots[key].append(item)

slot_counts = {k: len(v) for k, v in price_slots.items()}
print(f"Slots populated: {len(slot_counts)}")


In [ ]:
# ========== 价格均衡抽样：高价全收，低价桶限流并偏向非 Automotive ==========

# 固定种子，保证可复现
random.seed(123)
np.random.seed(123)
balanced_bundle = []

for price in range(1, 1000):
    bucket = price_slots.get(price, [])

    # 高价区样本少：整桶收下，缓解「便宜货过多」
    if price >= 240:
        balanced_bundle.extend(bucket)

    # 中低价且桶不太大：也整桶收下
    elif len(bucket) <= 1200:
        balanced_bundle.extend(bucket)

    else:
        # 桶太大：加权抽样最多 1200 条；Automotive 权重 1，其它 5（更偏向多样品类）
        weights = np.array([1 if item.category == "Automotive" else 5 for item in bucket], dtype=float)
        weights /= weights.sum()
        indices = np.random.choice(len(bucket), size=1200, replace=False, p=weights)
        for idx in indices:
            balanced_bundle.append(bucket[idx])

print(f"Balanced bundle size: {len(balanced_bundle):,}")


In [ ]:
# ========== 检查均衡后价格与品类是否更健康 ==========

bundle_prices = [item.price for item in balanced_bundle]
bundle_tokens = [item.token_count for item in balanced_bundle]
bundle_categories = Counter(item.category for item in balanced_bundle)
display(pd.Series(bundle_prices).describe())
display(pd.DataFrame.from_dict(bundle_categories, orient="index", columns=["count"]).sort_values("count", ascending=False))


In [ ]:
# ========== 可视化：价格直方图 + token 直方图 ==========

plt.figure(figsize=(12, 5))
plt.hist(bundle_prices, bins=range(0, 1000, 10), color="midnightblue", rwidth=0.8)
plt.xlabel("Price")
plt.ylabel("Count")
# 第二张图：token 长度分布（影响上下文是否被截断）
plt.figure(figsize=(12, 5))
plt.hist(bundle_tokens, bins=range(0, 300, 10), color="forestgreen", rwidth=0.8)
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()


In [ ]:
# ========== 打乱后划分 train / test ==========

random.seed(123)
random.shuffle(balanced_bundle)
# 测试集约 5%（上限 2000）；训练集约剩余（上限 40 万）
test_target = min(2000, max(1, len(balanced_bundle) // 20))
train_target = min(400_000, len(balanced_bundle) - test_target)
train_items = balanced_bundle[:train_target]
test_items = balanced_bundle[train_target:train_target + test_target]
print(f"Training set: {len(train_items):,}")
print(f"Test set: {len(test_items):,}")


In [ ]:
# ========== 抽出文本与价格列（训练用完整 prompt，测试用 test_prompt） ==========

train_prompts = [item.prompt for item in train_items]
train_prices = [item.price for item in train_items]
# test_prompt：不含价格数字，适合留给模型填
test_prompts = [item.test_prompt() for item in test_items]
test_prices = [item.price for item in test_items]


In [ ]:
# ========== 包装成 Hugging Face DatasetDict ==========

train_dataset = Dataset.from_dict({"text": train_prompts, "price": train_prices})
test_dataset = Dataset.from_dict({"text": test_prompts, "price": test_prices})
# 统一入口：后面可 to_parquet / 继续加工
pricing_dataset = DatasetDict({"train": train_dataset, "test": test_dataset})


### 持久化保存

把 Item 列表（pickle）和 Dataset（parquet）落到 `data/`，避免每次重跑漫长加载。


In [ ]:
# ========== 用 pickle 保存 train/test 的 Item 对象列表 ==========

storage_dir = Path("data")
storage_dir.mkdir(exist_ok=True)

# 二进制写入：保留 Item 实例上的全部字段（prompt、features 等后续可再挂）
with open(storage_dir / "balanced_train.pkl", "wb") as f:
    pickle.dump(train_items, f)

with open(storage_dir / "balanced_test.pkl", "wb") as f:
    pickle.dump(test_items, f)


In [ ]:
# ========== 同步导出 parquet（便于用 datasets / pandas 快速重载） ==========

pricing_dataset["train"].to_parquet(storage_dir / "balanced_train.parquet")
pricing_dataset["test"].to_parquet(storage_dir / "balanced_test.parquet")


## 基线模型

先建立「不调用 LLM」的对照线：随机、均值、特征回归、NLP、森林——用来衡量后面微调到底提升了多少。


### 随机锚点基线

完全瞎猜 $1–$999，给出误差下限的「地板」参考。


In [ ]:
# ========== 基线 A：均匀随机猜价格 ==========

def stochastic_anchor(item):
    # 忽略商品内容，只在 1..999 随机取整数美元
    return random.randrange(1, 1000)

random.seed(123)
# 在前 250 条测试集上跑 Tester（彩色日志 + 散点图）
Tester.test(stochastic_anchor, test_items[:250])


### 全局均值基线

永远预测训练集平均价——比随机强一点，但仍完全不看商品文本。


In [ ]:
# ========== 基线 B：训练集全局均值 ==========

train_price_values = [item.price for item in train_items]
# 所有训练价格的算术平均
global_mean_price = sum(train_price_values) / len(train_price_values)

def global_mean_estimator(item):
    # 对任何商品都返回同一个常数
    return global_mean_price

Tester.test(global_mean_estimator, test_items[:250])


In [ ]:
# ========== 把 details 解析成 features 字典，供特征工程使用 ==========

def parse_features(raw):
    # 空 details → 空字典
    if not raw:
        return {}
    try:
        # details 常以 JSON 字符串形式存放
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}

# 给训练集每条挂上 .features
for item in train_items:
    item.features = parse_features(item.details)
# 测试集同样处理
for item in test_items:
    item.features = parse_features(item.details)


### 特征工程

从 `Item Weight`、`Best Sellers Rank`、`Brand` 等字段抽出数值特征，再喂给线性回归 / 随机森林。


In [ ]:
# ========== 从 features 推断重量（统一换算到 pounds） ==========

def infer_weight(item):
    payload = item.features.get("Item Weight")
    if not payload:
        return None

    # 期望形如 "12 pounds" / "8 ounces" …
    parts = payload.split(" ")
    amount = float(parts[0])
    unit = parts[1].lower()

    if unit == "pounds":
        return amount

    if unit == "ounces":
        return amount / 16

    if unit == "grams":
        return amount / 453.592

    if unit == "milligrams":
        return amount / 453592

    if unit == "kilograms":
        return amount / 0.453592

    # 特殊单位：hundredths pounds
    if unit == "hundredths" and len(parts) > 2 and parts[2].lower() == "pounds":
        return amount / 100

    return None


In [ ]:
# ========== 畅销排名均值 + 是否头部品牌 ==========

def infer_rank(item):
    payload = item.features.get("Best Sellers Rank")
    if not payload:
        return None

    # 可能是 {类目: 排名} 字典；取所有排名的平均
    values = list(payload.values()) if isinstance(payload, dict) else []
    if not values:
        return None

    return sum(values) / len(values)

# 人为定义的「大牌」小写集合（拼写按原代码保留，含 nvidea）
top_brands = {"nvidea","hp","dell","lenovo","samsung","asus","sony","canon","apple","intel"}

def is_top_brand(item):
    brand = item.features.get("Brand")
    # 命中返回 1，否则 0（作为 0/1 特征）
    return 1 if brand and brand.lower() in top_brands else 0


In [ ]:
# ========== 用训练集统计「缺省填充值」：平均重量 / 平均排名 ==========

train_weights = [infer_weight(item) for item in train_items]
# 去掉 None 再平均；若全空则退回 1.0
train_weights = [value for value in train_weights if value is not None]
average_weight = sum(train_weights) / len(train_weights) if train_weights else 1.0
train_ranks = [infer_rank(item) for item in train_items]
train_ranks = [value for value in train_ranks if value is not None]
# 排名缺省用很大的数（表示「很不畅销」）
average_rank = sum(train_ranks) / len(train_ranks) if train_ranks else 1_000_000.0


In [ ]:
# ========== 组装单条样本的表格特征字典 ==========

def build_features(item):
    weight = infer_weight(item)
    rank = infer_rank(item)

    return {
        # 缺失则用训练集均值填充
        "weight": weight if weight is not None else average_weight,
        "rank": rank if rank is not None else average_rank,
        # 文本长度：用 test_prompt 字符数当粗糙信号
        "text_length": len(item.test_prompt()),
        "top_brand": is_top_brand(item)
    }


In [ ]:
# ========== 构建 train/test 特征表（测试侧先取前 250 条对齐 Tester） ==========

train_frame = pd.DataFrame([build_features(item) for item in train_items])
train_frame["price"] = [item.price for item in train_items]
test_frame = pd.DataFrame([build_features(item) for item in test_items[:250]])
test_frame["price"] = [item.price for item in test_items[:250]]


In [ ]:
# ========== 基线 C：线性回归（表格特征） ==========

feature_columns = ["weight", "rank", "text_length", "top_brand"]
X_train = train_frame[feature_columns]
y_train = train_frame["price"]
X_test = test_frame[feature_columns]
y_test = test_frame["price"]
linear_model = LinearRegression()
# 拟合：学习特征 → 价格 的线性映射
linear_model.fit(X_train, y_train)

def linear_baseline(item):
    # 单条预测：先 build_features → DataFrame → predict → float
    return float(linear_model.predict(pd.DataFrame([build_features(item)]))[0])

Tester.test(linear_baseline, test_items[:250])


### NLP 基线

不靠结构化字段，直接从 `test_prompt` 文本学价格：词袋回归与 Word2Vec + SVR。


In [ ]:
# ========== 基线 D：CountVectorizer 词袋 + 线性回归 ==========

document_texts = [item.test_prompt() for item in train_items]
price_targets = np.array([item.price for item in train_items])
# 最多 1000 维词袋，去掉英文停用词
vectorizer = CountVectorizer(max_features=1000, stop_words="english")
X_matrix = vectorizer.fit_transform(document_texts)
bow_model = LinearRegression()
bow_model.fit(X_matrix, price_targets)

def bow_predictor(item):
  # 注意：本函数体保持原缩进（两空格）；预测后用 max(...,0) 避免负价格
  pred = float(bow_model.predict(vectorizer.transform([item.test_prompt()]))[0])
  return max(pred, 0)

Tester.test(bow_predictor, test_items[:250])


In [ ]:
# ========== 基线 E：Word2Vec 文档向量 + LinearSVR ==========

# 每篇文档先 simple_preprocess 成词列表
processed_docs = [simple_preprocess(text) for text in document_texts]
# 训练词向量：400 维，窗口 5
word2vec_model = Word2Vec(sentences=processed_docs, vector_size=400, window=5, min_count=1, workers=4)

def document_vector(text):
    """把文档里能查到的词向量取均值，作为文档嵌入；空则零向量。"""
    words = simple_preprocess(text)
    vectors = [word2vec_model.wv[word] for word in words if word in word2vec_model.wv]

    if not vectors:
        return np.zeros(word2vec_model.vector_size)
    return np.mean(vectors, axis=0)

# 训练集文档向量矩阵
w2v_features = np.array([document_vector(text) for text in document_texts])
svr_model = LinearSVR()
svr_model.fit(w2v_features, price_targets)

def w2v_predictor(item):
    return float(svr_model.predict([document_vector(item.test_prompt())])[0])

Tester.test(w2v_predictor, test_items[:250])


In [ ]:
# ========== 基线 F：随机森林（同一套表格特征） ==========

forest_model = RandomForestRegressor(n_estimators=200, random_state=123)
forest_model.fit(X_train, y_train)

def forest_predictor(item):
    # 显式按 feature_columns 取列，避免列顺序问题
    return float(forest_model.predict(pd.DataFrame([build_features(item)])[feature_columns])[0])

Tester.test(forest_predictor, test_items[:250])


In [ ]:
# ========== 为 OpenAI 微调截取小子集（控制费用） ==========

# 前 200 条做训练，紧接着 50 条做验证
fine_tune_train = train_items[:200]
fine_tune_validation = train_items[200:250]


In [ ]:
# ========== 组装 Chat messages（system / user / assistant） ==========

def compose_messages(item, include_price=True):
    # system：约束「只回价格」——英文字符串保持原样
    system_message = "You estimate prices of items. Reply only with the price"
    # user：用 test_prompt 并去掉部分套话，减少与 PREFIX 重复
    user_prompt = item.test_prompt().replace(" to the nearest dollar", "").replace("\n\nPrice is $", "")
    # 训练时 assistant 给真值；推理时只留 "Price is $" 前缀让模型续写
    assistant_content = f"Price is ${item.price:.2f}" if include_price else "Price is $"
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_content}
    ]


In [ ]:
# ========== 把 Item 列表编码为 JSONL 文本（每行一条 messages） ==========

def build_jsonl(items):
    # 逐条收集 JSON 字符串
    lines = []
    for item in items:
        # OpenAI fine-tune 行格式：顶层键 messages
        payload = {"messages": compose_messages(item)}
        # 序列化成单行 JSON
        lines.append(json.dumps(payload))

    # 行与行之间用换行连接 → JSONL
    return "\n".join(lines)


In [ ]:
# ========== 写出训练 / 验证 JSONL 到 data/ ==========

train_jsonl = storage_dir / "balanced_pricer_train.jsonl"
validation_jsonl = storage_dir / "balanced_pricer_validation.jsonl"
train_jsonl.write_text(build_jsonl(fine_tune_train))
validation_jsonl.write_text(build_jsonl(fine_tune_validation))


In [ ]:
# ========== 上传 JSONL 到 OpenAI Files（purpose=fine-tune） ==========

openai_client = OpenAI()

with open(train_jsonl, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

with open(validation_jsonl, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")

# 笔记本里直接展示两个 File 对象，便于确认 id
train_file, validation_file


In [ ]:
# ========== 创建 Fine-Tuning Job（可选挂 W&B 集成） ==========

# Weights & Biases 集成配置：把训练曲线打到 project=balanced-pricer
wandb_integration = {"type": "wandb", "wandb": {"project": "balanced-pricer"}}
fine_tune_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    # 基座模型 id 保持原样
    model="gpt-4o-mini-2024-07-18",
    seed=123,
    hyperparameters={"n_epochs": 1},
    integrations=[wandb_integration],
    suffix="balanced-pricer"
)
# 展示 job 对象（含 id / status）
fine_tune_job


In [ ]:
# ========== 查询任务状态与最近事件 ==========

job_status = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
job_events = openai_client.fine_tuning.jobs.list_events(fine_tuning_job_id=fine_tune_job.id, limit=10)
job_status, job_events


In [ ]:
# ========== 取出微调完成后的模型名 ==========

fine_tuned_model_name = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id).fine_tuned_model
print(fine_tuned_model_name)


In [ ]:
# ========== 微调模型预测函数：调用 Chat Completions 再解析价格 ==========

def tuned_predictor(item):
    # include_price=False → assistant 只有 "Price is $"，让模型补全数字
    messages = compose_messages(item, include_price=False)
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages,
        seed=123,
        max_tokens=7
)
    answer = response.choices[0].message.content
    # 用前面的 get_price 从文本抠 float
    return get_price(answer)


In [ ]:
# ========== 冒烟测试：打印第一条真值与模型预测 ==========

if test_items:
    sample_item = test_items[0]
    print(sample_item.price)
    print(tuned_predictor(sample_item))


In [ ]:
# ========== 最终评测：微调模型 vs 前面各基线（同一 Tester 协议） ==========

Tester.test(tuned_predictor, test_items[:250])
